# 06 · A feed-forward network with DeepNetts

Logistic regression draws a straight decision boundary. Add hidden layers and the
network can bend it — at the cost of more parameters to train.

Same pipeline, same data, same metric; only the architecture changes.


In [0]:
addDependency("com.deepnetts:deepnetts-core:4.0.4");

import deepnetts.net.FeedForwardNetwork;
import deepnetts.net.layers.activation.ActivationType;
import deepnetts.net.loss.LossType;
import deepnetts.data.DataSets;
import deepnetts.data.MLDataItem;
import deepnetts.data.TabularDataSet;
import deepnetts.data.TrainTestSplit;
import deepnetts.data.norm.MaxScaler;
import javax.visrec.ml.eval.EvaluationMetrics;
import java.nio.file.Files;
import java.nio.file.Path;

// The notebook lives in notebooks/exercises/, so walk up to find the repo's
// data/ directory - the same trick the other notebooks use.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/credit_card_fraud_imbalanced.csv");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/credit_card_fraud_imbalanced.csv not found above " + cwd);
}


## 1 · Load and split

Same credit-card transactions, same 7 → 1 shape. Split first, so the test set is
never seen during training.


In [0]:
TabularDataSet<MLDataItem> dataSet = DataSets.readCsv(dataset().toString(), 7, 1, true);

TrainTestSplit split = DataSets.trainTestSplit(dataSet, 0.8);
var trainSet = split.getTrainingSet();
var testSet = split.getTestSet();
println("loaded %d rows — train %d   test %d", dataSet.size(), trainSet.size(), testSet.size());


## 2 · Scale


In [0]:
MaxScaler scaler = DataSets.scaleToMax(trainSet);
scaler.apply(testSet);
println("scaled to [0, 1]");


## 3 · The architecture

Two hidden layers (32 and 16 ReLU units) between the inputs and the sigmoid
output.


In [0]:
FeedForwardNetwork neuralNet = FeedForwardNetwork.builder()
        .addInputLayer(7)
        .addFullyConnectedLayer(32, ActivationType.RELU)
        .addFullyConnectedLayer(16, ActivationType.RELU)
        .addOutputLayer(1, ActivationType.SIGMOID)
        .lossFunction(LossType.CROSS_ENTROPY)
        .build();

neuralNet.getTrainer()
        .setLearningRate(0.01f)
        .setStopError(0.13f)
        .setStopEpochs(5)
        .setShuffle(true);


In [0]:
// DeepNetts logs through java.util.logging; route it into the notebook.
class InternalLogHandler extends java.util.logging.Handler {
    public void close() {}
    public void flush() {}
    public void publish(java.util.logging.LogRecord logRecord) {
        println(logRecord.getMessage());
    }
}

deepnetts.core.DeepNetts.LOGGER.addHandler(new InternalLogHandler());


## 4 · Train


In [0]:
neuralNet.train(trainSet);


## 5 · Evaluate


In [0]:
EvaluationMetrics evaluation = neuralNet.test(testSet);
println(evaluation);


## 6 · Save the trained model

DeepNetts serialises the whole network — architecture, weights, scaling — to a
`.dnet` file next to the notebook.


In [0]:
neuralNet.save(cwd.resolve("fraudDetectionNet.dnet").toString());
println("saved fraudDetectionNet.dnet");


## 7- Use network for prediction

In [0]:
var prediction = neuralNet.predict(testSet.get(0).getInput());
println("Fraud probability: "+prediction)